# RO-049 Moving through unknown maps: every number in the Note

The planners live in `images/replan.py`, on chapter RO-04's room grid (`images/searchkit.py`: 0.5 m cells, 76 free,
start cell S, goal cell G under the door, 8-connected moves of 0.5 m and 0.707 m that never cut the corner of a
blocked cell). A box of 1.0 m x 0.5 m stands at x 2.0 to 3.0 m, y 2.0 to 2.5 m; the map does not show it, and the
robot sees it within 0.5 m. This notebook runs each method and prints the numbers the Note quotes. Nothing here is
random, so there is no seed to set.

In [1]:
import sys, math
sys.path.insert(0, 'images')
from replan import *
r3 = lambda x: round(x, 3)
plen = lambda p, h=0.5: sum(h * math.dist(a, b) for a, b in zip(p, p[1:]))
gr = Grid(1)
print('free cells', int(gr.base.sum()), ' box cells', sorted(gr.box_cells()))

free cells 76  box cells [(4, 4), (5, 4)]


## 1. The drive with the box, replanning with A* from scratch (Overview, Figure 1)

In [2]:
V0 = dijkstra_from_goal(gr)
R = drive(1, from_full=True)
E = R['events'][0]
print('robot sees the box at', E['pos'], 'after move', E['step'], ' new blocked cells', E['new'])
print('its number before', r3(V0[E['pos']]), ' after', r3(E['g']))
print('A* from scratch there:', len(E['astar']), 'cells, route', r3(E['a_cost']), 'm', E['a_path'])
print('driven in all', r3(R['length']), 'm;  planned', r3(V0[START]), 'm')

robot sees the box at (5, 3) after move 1  new blocked cells [(4, 4), (5, 4)]
its number before 2.707  after 4.414
A* from scratch there: 11 cells, route 4.414 m [(5, 3), (4, 3), (3, 3), (3, 4), (3, 5), (4, 6), (5, 7), (6, 7), (7, 7)]
driven in all 5.121 m;  planned 3.414 m


## 2. D* Lite repairing the numbers (Figure 2)

In [3]:
g = Grid(1)
D = DStarLite(g, (5, 3), GOAL, g=dijkstra_from_goal(g))
box = sorted(g.box_cells()); g.seen.update(box)
D.blocked(box)
print('inconsistent after the box is seen:', sorted(D.inU))
for c in sorted(D.inU):
    print('  ', c, 'number', r3(D.G(c)) if D.G(c) < INF else 'inf', ' lookahead', r3(D.R(c)) if D.R(c) < INF else 'inf')
n = D.compute()
print('cells taken off the queue:', n)
for c, kind, old, new, key in D.log:
    print('  ', c, kind, r3(old) if old < INF else 'inf', '->', r3(new) if new < INF else 'inf')
print('robot cell', r3(D.G((5, 3))), ' still queued', len(D.inU), sorted(D.inU))
print('distinct cells changed', len({c for c, *_ in D.log}), ' never touched', len(V0) - len({c for c, *_ in D.log}))
print('lookahead of the robot cell before repair, neighbours:')
g2 = Grid(1); g2.seen.update(box)
for n_, s in g2.nbrs((5, 3)):
    if s < INF: print('  ', n_, r3(s), '+', r3(V0[n_]), '=', r3(s + V0[n_]))

inconsistent after the box is seen: [(3, 4), (4, 3), (4, 4), (5, 3), (5, 4)]
   (3, 4) number 2.621  lookahead 2.914
   (4, 3) number 2.914  lookahead 3.207
   (4, 4) number 2.414  lookahead inf
   (5, 3) number 2.707  lookahead 3.414
   (5, 4) number 2.207  lookahead inf
cells taken off the queue: 12
   (5, 4) raise 2.207 -> inf
   (5, 3) raise 2.707 -> inf
   (4, 4) raise 2.414 -> inf
   (4, 3) raise 2.914 -> inf
   (5, 2) raise 3.207 -> inf
   (3, 4) raise 2.621 -> inf
   (3, 4) lower inf -> 2.914
   (3, 3) raise 3.121 -> inf
   (4, 2) raise 3.414 -> inf
   (3, 3) lower inf -> 3.414
   (4, 3) lower inf -> 3.914
   (5, 3) lower inf -> 4.414
robot cell 4.414  still queued 8 [(2, 3), (2, 4), (3, 1), (3, 2), (4, 1), (4, 2), (5, 1), (5, 2)]
distinct cells changed 8  never touched 68
lookahead of the robot cell before repair, neighbours:
   (4, 2) 0.707 + 3.414 = 4.121
   (4, 3) 0.5 + 2.914 = 3.414
   (5, 2) 0.5 + 3.207 = 3.707


## 3. How much repairing saves, at three cell sizes (Figure 3)

In [4]:
for k in (1, 2, 5):
    r = drive(k); ev = r['events']
    print(f'cell {0.5 / k} m: free cells', int(r['grid'].base.sum()), ' box cells', len(r['truth']),
          ' first plan D* Lite', r['first'][0], 'A*', r['first'][1], ' replans', len(ev),
          ' D* Lite', sum(e['dstar'] for e in ev), ' A*', sum(len(e['astar']) for e in ev), ' driven', r3(r['length']))
    if k == 5:
        print('   per replan (D* Lite, A*):', [(e['dstar'], len(e['astar'])) for e in ev])

cell 0.5 m: free cells 76  box cells 2  first plan D* Lite 12 A* 9  replans 1  D* Lite 19  A* 11  driven 5.121
cell 0.25 m: free cells 304  box cells 8  first plan D* Lite 32 A* 26  replans 2  D* Lite 67  A* 73  driven 4.475


cell 0.1 m: free cells 1900  box cells 50  first plan D* Lite 134 A* 117  replans 7  D* Lite 387  A* 596  driven 4.087
   per replan (D* Lite, A*): [(238, 125), (44, 125), (39, 105), (35, 117), (31, 90), (0, 17), (0, 17)]


## 4. Real-time search: LRTA* and RTAA* (Figure 4)

In [5]:
route, log = lrta()
print('LRTA*:', len(route) - 1, 'moves,', r3(plen(route)), 'm;', len(route) - 1, 'cells looked at, one per move')
for L in log:
    print('  at', L['pos'], ' h', r3(L['h_old']), '->', r3(L['h_new']), ' best', [(r3(v), n) for v, _, n in L['cand'][:3]])
for N in (1, 2, 4, 8):
    rr, tot, lg = rtaa(N)
    print('RTAA* N =', N, ':', len(rr) - 1, 'moves,', r3(plen(rr)), 'm,', tot, 'cells in', len(lg), 'searches')
rr, tot, lg = rtaa(4)
for L in lg:
    print('  search at', L['pos'], ' taken off', L['closed'], ' best open', L['best'], ' f', r3(L['f']))
    for u, (o, nw) in L['upd'].items():
        print('     ', u, 'g', r3(L['g'][u]), ' h', r3(o), '->', r3(nw))

LRTA*: 9 moves, 5.121 m; 9 cells looked at, one per move
  at (4, 2)  h 3.121 -> 3.121  best [(3.121, (5, 3)), (3.121, (4, 3)), (3.414, (5, 2))]
  at (5, 3)  h 2.414 -> 3.121  best [(3.121, (4, 3)), (3.414, (5, 2)), (3.828, (4, 2))]
  at (4, 3)  h 2.621 -> 3.328  best [(3.328, (3, 3)), (3.621, (5, 2)), (3.621, (4, 2))]
  at (3, 3)  h 2.828 -> 3.121  best [(3.121, (3, 4)), (3.828, (2, 4)), (3.828, (4, 2))]
  at (3, 4)  h 2.621 -> 2.914  best [(2.914, (3, 5)), (3.621, (2, 4)), (3.621, (3, 3))]
  at (3, 5)  h 2.414 -> 2.414  best [(2.414, (4, 6)), (2.414, (4, 5)), (2.707, (3, 6))]
  at (4, 6)  h 1.707 -> 1.707  best [(1.707, (5, 7)), (1.707, (5, 6)), (2.0, (4, 7))]
  at (5, 7)  h 1.0 -> 1.0  best [(1.0, (6, 7)), (1.414, (6, 6)), (1.707, (5, 6))]
  at (6, 7)  h 0.5 -> 0.5  best [(0.5, (7, 7)), (1.207, (7, 6)), (1.207, (6, 6))]
RTAA* N = 1 : 9 moves, 5.121 m, 9 cells in 9 searches
RTAA* N = 2 : 10 moves, 5.828 m, 14 cells in 7 searches
RTAA* N = 4 : 8 moves, 4.414 m, 13 cells in 4 searches


## 5. Bug 1 and Bug 2 (Figure 5)

In [6]:
B = bug_paths()
print('straight line start to door', r3(B['d']), ' obstacle perimeter', r3(B['perimeter']))
print('hit point', tuple(round(v, 3) for v in B['hit']), ' start to hit', r3(math.dist(BUG_START, B['hit'])),
      ' hit to door', r3(math.dist(B['hit'], BUG_GOAL)))
print('Bug 1: closest point', tuple(round(v, 3) for v in B['close']), r3(B['close_dist']), 'm from the door, at',
      r3(B['s_close']), 'm round from the hit point; length', r3(B['len1']))
print('   bound d + 1.5 p =', r3(B['d'] + 1.5 * B['perimeter']))
print('Bug 2: leave point', tuple(round(v, 3) for v in B['exit']), r3(math.dist(B['exit'], BUG_GOAL)),
      'm from the door, after', r3(B['s_leave']), 'm of boundary; length', r3(B['len2']))
print('   bound d + n p / 2 with n = 2 crossings =', r3(B['d'] + B['perimeter']))
r4 = lambda x: round(x, 4)
h = math.dist(BUG_START, B['hit'])
print('Bug 1 parts to 4 decimals:', r4(h), r4(B['perimeter']), r4(B['s_close']), r4(B['close_dist']), '=', r4(B['len1']))
print('Bug 2 parts to 4 decimals:', r4(h), r4(B['s_leave']), r4(math.dist(B['exit'], BUG_GOAL)), '=', r4(B['len2']))
print('1.5 p =', r3(1.5 * B['perimeter']))

straight line start to door 3.4  obstacle perimeter 11.702
hit point (2.4, 1.75)  start to hit 0.85  hit to door 2.55
Bug 1: closest point (3.524, 3.249) 0.755 m from the door, at 3.604 m round from the hit point; length 16.912
   bound d + 1.5 p = 20.953
Bug 2: leave point (2.933, 2.75) 1.417 m from the door, after 2.619 m of boundary; length 4.885
   bound d + n p / 2 with n = 2 crossings = 15.102
Bug 1 parts to 4 decimals: 0.85 11.7023 3.6043 0.755 = 16.9116
Bug 2 parts to 4 decimals: 0.85 2.6187 1.4167 = 4.8853
1.5 p = 17.553
